In [ ]:
from fastapi import FastAPI, HTTPException
from pydantic import BaseModel
from typing import Optional, Any
import mlflow
import re
from bs4 import BeautifulSoup

app = FastAPI(title="Rakuten Product Classification API")

# MLflow
mlflow.set_tracking_uri("file:///C:/Users/user/Rakuten-Challenge/mlruns")
model = mlflow.pyfunc.load_model("models:/rakuten_model/Production")


class Product(BaseModel):
    designation: str
    description: Optional[str] = ""


def safe_to_str(x: Any) -> str:
    if x is None:
        return ""
    return str(x)


def clean_text(text: Any) -> str:
    text = safe_to_str(text)
    text = BeautifulSoup(text, "html.parser").get_text()
    text = text.lower()
    text = re.sub(r"[^a-zàâçéèêëîïôûùüÿñæœ0-9 ]", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text


@app.get("/health")
def health():
    return {"status": "ok"}


@app.post("/predict")
def predict(product: Product):
    raw = f"{product.designation} {product.description}".strip()
    text = clean_text(raw)

    if not text:
        raise HTTPException(status_code=400, detail="Text is empty after preprocessing")

    try:
        pred = model.predict([text])[0]  # list[str] uniquement
        return {"prediction": int(pred)}
    except Exception as e:
        raise HTTPException(status_code=500, detail=f"ML error ({type(e).__name__}): {e}")


c:\Users\user\anaconda3\Lib\site-packages\mlflow\store\artifact\utils\models.py:31: FutureWarning: ``mlflow.tracking.client.MlflowClient.get_latest_versions`` is deprecated since 2.9.0. Model registry stages will be removed in a future major release. To learn more about the deprecation of model registry stages, see our migration guide here: https://mlflow.org/docs/latest/model-registry.html#migrating-from-stages
  latest = client.get_latest_versions(name, None if stage is None else [stage])
